### **Chapter 9: Data-Enabled Predictive Control (DeePC)**

In this chapter, we implement a minimal deterministic **Data-Enabled Predictive Control (DeePC)** controller for the linear Mountain Car cases. The closed-loop workflow is intentionally kept close to the MPC implementation from Chapter 5:

- MPC predicts trajectories with an explicit dynamics model.
- DeePC predicts trajectories directly from one persistently exciting input-output dataset.
- Both solve a finite-horizon optimization problem at every time step and apply only the first optimal input.

We use the full state as the output, $y=x=[p,v]^\top$, and OSQP to solve the DeePC quadratic program.

In [ ]:
import sys
import os
import time
import numpy as np

sys.path.append(os.path.abspath(".."))
from utils.env import *
from utils.controller import *
from utils.simulator import *
from ex9_DeePC.deepc_utils import *

### **Problem Setup**

We use the same Mountain Car environment as in Chapter 5. For this first DeePC implementation, use only:

- `case = 1`: flat terrain, or
- `case = 2`: constant slope.

For the constant-slope case, the controller internally works in deviation coordinates around the target equilibrium $(x_{\mathrm{ref}},u_{\mathrm{eq}})$. This removes the constant gravity offset and gives an LTI input-output behavior suitable for the basic DeePC formulation.

In [ ]:
# Define profile of slope, the initial / target state
case = 2 # use 1 (flat) or 2 (constant slope)
initial_position = -0.5
initial_velocity = 0.0
target_position = 0.6
target_velocity = 0.0

# For case 2, the equilibrium input is nonzero, so use wider input bounds.
if case == 1:
    input_lbs = -1.0
    input_ubs = 1.0
else:
    input_lbs = -2.5
    input_ubs = 2.5

env = Env(
    case,
    np.array([initial_position, initial_velocity]),
    np.array([target_position, target_velocity]),
    input_lbs=input_lbs,
    input_ubs=input_ubs,
)
dynamics = Dynamics(env)

env.test_env()

<br>

----

### **Part 1: Offline Data Collection**

DeePC replaces the explicit prediction model with a measured trajectory. We collect one open-loop dataset

$$
\{u_0^d,\ldots,u_{T-1}^d\}, \qquad \{y_0^d,\ldots,y_T^d\},
$$

using a random excitation around the equilibrium input. The offline excitation must be sufficiently persistently exciting. The controller checks this rank condition during setup.

This offline dataset is different from the short online history used to initialize each DeePC solve.

In [ ]:
# Data collection parameters
freq = 20
n_data = 400
excitation_amplitude = 0.8

u_data, y_data = collect_deepc_data(
    env,
    dynamics,
    freq=freq,
    n_samples=n_data,
    excitation_amplitude=excitation_amplitude,
    initial_state=env.target_state,
    seed=1,
)

print("u_data shape:", u_data.shape)
print("y_data shape:", y_data.shape)
print("equilibrium input:", dynamics.get_equilibrium_input(env.target_state))

In [ ]:
# Optional: inspect the collected trajectory
fig, ax = plt.subplots(1, 2, figsize=(12, 3))
ax[0].plot(np.arange(n_data) / freq, u_data[:, 0])
ax[0].set_xlabel("Time (s)")
ax[0].set_ylabel("Input u")
ax[0].set_title("Offline excitation")

ax[1].plot(np.arange(n_data + 1) / freq, y_data[:, 0], label="position")
ax[1].plot(np.arange(n_data + 1) / freq, y_data[:, 1], label="velocity")
ax[1].set_xlabel("Time (s)")
ax[1].set_title("Offline state/output data")
ax[1].legend()
plt.tight_layout()
plt.show()

<br>

----

### **Part 2: DeePC Formulation**

For a prediction horizon $N$ and an initialization horizon $T_{\mathrm{ini}}$, the offline trajectory is partitioned into Hankel matrices

$$
\begin{bmatrix}U_p\\U_f\end{bmatrix},\qquad
\begin{bmatrix}Y_p\\Y_f\end{bmatrix}.
$$

At each control step, the most recent input-output history is imposed through

$$
U_pg=u_{\mathrm{ini}}, \qquad Y_pg=y_{\mathrm{ini}},
$$

and the future trajectory is generated directly from data:

$$
u=U_fg, \qquad y=Y_fg.
$$

The controller solves a quadratic program in the single decision vector $g$:

$$
\min_g \; \|Y_fg\|_{\bar Q}^2 + \|U_fg\|_{\bar R}^2 + \lambda_g\|g\|_2^2,
$$

subject to the history, current-state, state, and input constraints.

The implementation is in `ex9_DeePC/deepc_utils.py`. Its public interface mirrors the Chapter 5 MPC controller: `compute_action()` returns the first input together with predicted state and input trajectories.

In [ ]:
# DeePC cost and horizon parameters
Q = np.diag([1.0, 1.0])
R = np.array([[0.1]])
Qf = Q

N = 20
T_ini = 4
lambda_g = 1e-6

controller_deepc = DeePCController(
    env,
    dynamics,
    u_data,
    y_data,
    Q,
    R,
    Qf,
    freq,
    N,
    T_ini=T_ini,
    lambda_g=lambda_g,
    history_initialization='equilibrium', # alternatively: 'zero'
    name='DeePC_0',
    verbose=False,
)

print("Up:", controller_deepc.Up.shape)
print("Yp:", controller_deepc.Yp.shape)
print("Uf:", controller_deepc.Uf.shape)
print("Yf:", controller_deepc.Yf.shape)

#### **Initialization**

The online past trajectory is initialized in one function:

```python
controller_deepc.initialize_history(current_state, mode='equilibrium')
```

- `equilibrium`: repeats the current state and the corresponding equilibrium input.
- `zero`: repeats the current state and zero input; this is mainly appropriate when zero input is an equilibrium input, as in the flat-terrain case.

In normal use, no manual call is needed: `compute_action()` initializes the history automatically on its first call and then maintains the receding history internally.

<br>

----

### **Part 3: Receding-Horizon DeePC Simulation**

The simulation is intentionally identical to Chapter 5. At every time step:

1. update the latest measured state,
2. solve the DeePC QP,
3. apply only the first optimal input,
4. shift the past input-output window,
5. repeat at the next state measurement.

In [ ]:
# Define simulation time
t_terminal = 8

start_time = time.time()

# Same Simulator / Visualizer workflow as MPC in Chapter 5
simulator_deepc = Simulator(dynamics, controller_deepc, env, 1/freq, t_terminal)
simulator_deepc.run_simulation()

print(f"Average computation time per step: {(time.time()-start_time) / (freq * t_terminal):.6f} s")

visualizer_deepc = Visualizer(simulator_deepc)
visualizer_deepc.display_plots()
visualizer_deepc.display_animation()

#### **Next Demonstrative Cases**

This first notebook intentionally keeps the controller implementation minimal. The following notebooks isolate the main properties of DeePC:

- `9.2_fundamental_lemma.ipynb`: unseen-trajectory reconstruction, persistent excitation, and PE failure;
- `9.3_deepc_vs_mpc.ipynb`: deterministic LTI DeePC--MPC equivalence;
- `9.4_history_as_state.ipynb`: position-only output and past I/O as an implicit state;
- `9.5_robust_deepc.ipynb`: bumpy-terrain model mismatch, loss of exact Hankel representation, and regularized DeePC.

The reusable implementation remains in `deepc_utils.py`; experiment-specific sweeps and plots stay inside the notebooks so that the controller remains easy to modify.